# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/udayk-25/FLYRANK/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

### Data Contract Answers:
1. **Unit of Analysis**: One row = **One unique webpage (pseudonymized content item) associated with a client (`content_id`, `client_id`)**.
2. **Time Window**: Features are computed as aggregations over a **90-day historical window** of Google Search Console (GSC) and Google Analytics (GA4) logs. The predictive target label is computed over the **subsequent 30-day post-prediction window** (or trend direction based on historical decay) to prevent look-ahead bias.
3. **Table(s) Used**: We utilize `fact_content_daily_performance` for performance signals and `dim_content` for content age/metadata.
4. **Target (Label/Proxy)**: A binary label `is_declining` indicating whether the webpage organic traffic (clicks/impressions) decayed by more than 20% compared to its previous performance.
5. **Deliberate Exclusion**: We exclude the `trend_direction` and `trend_pct` fields from model training, as these directly leak the future traffic outcome.


In [1]:
# No code required here; contract definitions are in the markdown cell above.


## 2. Fields: feature / label / context / excluded

We classify every field in the dataset into one of the four contract buckets:

- **Feature** (Safe to use, knowable before prediction):
  - `impressions_90d`, `clicks_90d`, `pageviews_90d`, `sessions_90d`, `ctr`, `avg_position`, `days_since_last_update`, `word_count`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`.
- **Label / Proxy** (What we predict):
  - `is_declining_label` (derived from `trend_direction` == 'down').
- **Context** (For grouping, joining, and reading — never passed to model):
  - `content_id`, `client_id`, `content_type`, `main_intent`.
- **Excluded** (Leaky or future outcome fields):
  - `trend_pct` (represents target traffic drop amount; leaks label).
  - `trend_direction` (categorical outcome; leaks label).


In [2]:
# No code required here; field classification is in the markdown cell above.


## 3. Verify it with queries (grain, counts, missing values, windows)

Below, we connect DuckDB to query our local dataset slice (`data/raw/content_refresh_anonymized.csv`) to prove three key facts about the data contract:
1. **The Grain**: Prove that `(content_id, client_id)` is a unique key (returns 0 rows with count > 1).
2. **Total Counts**: Output the row distribution and base rate of the target label (`trend_direction`).
3. **Availability & Missingness**: Prove how many rows have missing GSC data (represented by `avg_position = 0` in this dataset).


In [3]:
import duckdb
import pandas as pd

# Establish DuckDB connection
con = duckdb.connect()

# Load our local data slice into pandas first
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# Register pandas DataFrame in DuckDB
con.register("df", df)

print("--- QUERY 1: Verification of the Grain (Unique Keys) ---")
grain_check = con.execute("""
    SELECT content_id, client_id, COUNT(*) as row_count
    FROM df
    GROUP BY 1, 2
    HAVING row_count > 1
    LIMIT 5
""").df()
print(f"Rows violating grain (should be empty):\n{grain_check}\n")

print("--- QUERY 2: Verification of Counts & Label Base Rates ---")
counts_check = con.execute("""
    SELECT trend_direction, 
           COUNT(*) as row_count,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) as percentage
    FROM df
    GROUP BY 1
    ORDER BY row_count DESC
""").df()
print(f"{counts_check}\n")

print("--- QUERY 3: Verification of GSC Missing Data (avg_position = 0) ---")
missingness_check = con.execute("""
    SELECT COUNT(*) as total_rows,
           SUM(CASE WHEN avg_position = 0 THEN 1 ELSE 0 END) as missing_gsc_rows,
           ROUND(AVG(CASE WHEN avg_position = 0 THEN 1.0 ELSE 0 END) * 100.0, 2) as missing_percentage
    FROM df
""").df()
print(missingness_check)


--- QUERY 1: Verification of the Grain (Unique Keys) ---
Rows violating grain (should be empty):
Empty DataFrame
Columns: [content_id, client_id, row_count]
Index: []

--- QUERY 2: Verification of Counts & Label Base Rates ---
  trend_direction  row_count  percentage
0            down      16262       54.21
1          stable       5962       19.87
2              up       4388       14.63
3             new       2236        7.45
4            flat       1152        3.84

--- QUERY 3: Verification of GSC Missing Data (avg_position = 0) ---
   total_rows  missing_gsc_rows  missing_percentage
0       30000            1205.0                4.02


## 4. Data limits

### Feature Leakage Experiment:
To demonstrate the danger of label leakage, we build a 5-feature model (using `impressions_90d`, `ctr`, `avg_position`, `days_since_last_update`, and `word_count` as features) and run two scenarios:
1. **Leaky Scenario**: We add `trend_pct` directly to the features. The model F1-score jumps to a perfect 1.0 because the features contain the future label answer.
2. **Honest Scenario**: We remove `trend_pct` and train only on the 5 safe features, yielding an honest, realistic score.

### One Named Limitation:
**Unbalanced Client History**: GSC/GA4 history start dates vary across clients. Older pages for newer clients lack historical features, causing a cold-start bias where new pages are penalized as "freshness decay" simply due to missing timeline records.


In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import numpy as np

# Define the binary target label: is_declining = trend_direction == 'down'
df['is_declining'] = df['trend_direction'].str.lower().eq('down').astype(int)

# Scenario A: Leaky Model (including trend_pct)
leaky_cols = ['impressions_90d', 'ctr', 'avg_position', 'days_since_last_update', 'word_count', 'trend_pct']
df_clean = df.dropna(subset=leaky_cols)
X_leaky, y = df_clean[leaky_cols], df_clean['is_declining']

X_tr, X_te, y_tr, y_te = train_test_split(X_leaky, y, test_size=0.25, random_state=42, stratify=y)
leaky_model = RandomForestClassifier(n_estimators=100, max_depth=3, random_state=42).fit(X_tr, y_tr)
print("=== LEAKY MODEL PERFORMANCE (F1 = 1.0 expected due to leakage) ===")
print(classification_report(y_te, leaky_model.predict(X_te), digits=3))

# Scenario B: Clean Model (removing leaked feature)
honest_cols = ['impressions_90d', 'ctr', 'avg_position', 'days_since_last_update', 'word_count']
X_honest = df_clean[honest_cols]

X_tr_h, X_te_h, y_tr_h, y_te_h = train_test_split(X_honest, y, test_size=0.25, random_state=42, stratify=y)
honest_model = RandomForestClassifier(n_estimators=100, max_depth=3, random_state=42).fit(X_tr_h, y_tr_h)
print("\n=== HONEST MODEL PERFORMANCE (Clean, realistic performance) ===")
print(classification_report(y_te_h, honest_model.predict(X_te_h), digits=3))


=== LEAKY MODEL PERFORMANCE (F1 = 1.0 expected due to leakage) ===
              precision    recall  f1-score   support

           0      1.000     1.000     1.000      1622
           1      1.000     1.000     1.000      3170

    accuracy                          1.000      4792
   macro avg      1.000     1.000     1.000      4792
weighted avg      1.000     1.000     1.000      4792




=== HONEST MODEL PERFORMANCE (Clean, realistic performance) ===
              precision    recall  f1-score   support

           0      0.674     0.038     0.072      1622
           1      0.668     0.991     0.798      3170

    accuracy                          0.668      4792
   macro avg      0.671     0.514     0.435      4792
weighted avg      0.670     0.668     0.552      4792



## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
